# PEARL: Train + evaluate + calibrate (standalone)

No project upload needed: the code is embedded in this notebook (cell **A0**). Drive only needs:

```
MyDrive/thesis/
    pearl_venv_t4.tar.gz   <- your prebuilt venv
    data_gauss.zip         <- (or pearl_data.zip, data_srad.zip, ...) containing data/preprocessed/{figshare,pcosgen}/{train,val,test}
```

**One training run, two stages (per model):**

1. **Stage 1 - figshare**: train from ImageNet weights on figshare -> evaluate on figshare test (in-domain) and pcosgen test (cross-domain) -> calibration -> MC-Dropout uncertainty
2. **Stage 2 - pcosgen**: continue from the stage-1 weights and train on pcosgen (`pcosgen_ft`) -> evaluate on pcosgen test (in-domain) and figshare test -> calibration -> uncertainty

Stage 2 only runs after stage 1 of the same model has finished. `TRAIN_DATASETS` must stay `["figshare"]` - putting `"pcosgen"` there would start a *separate* pcosgen training from ImageNet weights, which is not the pipeline.

**Drive output**
```
thesis/models/<exp>/<figshare|pcosgen_ft>/<model>/    best.pt, config.yaml, final_metrics.json
thesis/results/<exp>/<figshare|pcosgen_ft>/<model>/   metrics.json, cross_<ds>.json, calibration/, uncertainty/, logs
```
`<exp>` is taken from the data zip name (`data_gauss.zip` -> `gauss`), so different preprocessings never overwrite each other.

**How to use**: pick a GPU runtime -> edit **A1** -> run **A1..A5** -> run **B**. Set `QUICK_TEST=True` first (a few minutes) to check everything works.
Re-running after a disconnect skips finished stages (weights are pulled back from Drive). Then use `pearl_ensemble_standalone.ipynb`.


## Setup (once per session)

In [37]:
# A0. Embedded PEARL code (src/ + configs/). Do not edit.
BUNDLE_B64 = "..."
print("bundle:", len(BUNDLE_B64) // 1024, "KB")


bundle: 129 KB


## Start From Here

### A1. Config

In [38]:
import builtins
all, any = builtins.all, builtins.any   # a leftover variable named `all` in the Colab kernel broke 'all(...)' ('list' object is not callable)

# ======================= EDIT HERE =======================
DRIVE_DIR     = "/content/drive/MyDrive/thesis"
DATA_ZIP      = "data_srad.zip"          # <- change to "data_srad.zip" (or any zip with data/preprocessed/...)
EXP_NAME      = None                       # None = auto from zip name (gauss / srad). Output folders use this.
VENV_ARCHIVE  = "pearl_venv_t4.tar.gz"

# ---- WHAT TO TRAIN ----
# One or more of: convnext_tiny, densenet169, efficientnet_b0, swin_tiny, vit_base   (or MODELS = all)
MODELS         = ["vit_base"]
# Stage 1 trains on these datasets from ImageNet weights. Keep this as ["figshare"]:
# pcosgen is NOT trained separately, it is stage 2 (fine-tune of the figshare model), see next line.
TRAIN_DATASETS = ["figshare"]
# Stage 2: continue the finished figshare model on pcosgen (saved as run "pcosgen_ft").
FINETUNE_FIGSHARE_TO_PCOSGEN = True
SKIP_TRAINED   = True          # skip runs already finished (here or on Drive). Safe to re-run after a disconnect.
QUICK_TEST     = False         # True = 1 epoch, 1 model, tiny analysis, nothing saved to Drive. RUN THIS FIRST.

# ---- per-model analysis ----
RUN_EVALUATE    = True         # test metrics, in-domain + cross-domain
RUN_CALIBRATION = True         # temperature scaling, ECE, reliability diagram
RUN_UNCERTAINTY = True         # MC-Dropout + referral curve
RUN_XAI         = False        # Grad-CAM (slow, off by default)
MC_PASSES       = 50
XAI_METHODS     = ["gradcam"]
XAI_SAMPLES     = 20

# ---- training settings (keep identical for all models/sessions so results are comparable) ----
EPOCHS    = 100                # early stopping (patience 20) still applies
LR        = 1e-4
SEED      = 42
PRECISION = "fp16"             # T4 has no bf16
BATCH_SIZES = {"convnext_tiny": 32, "densenet169": 32, "efficientnet_b0": 32, "swin_tiny": 32, "vit_base": 16}
NUM_WORKERS = None
FT_EPOCHS = None               # None = EPOCHS
FT_LR     = None               # None = LR
EXTRA_TRAIN_ARGS = []
# =========================================================
if QUICK_TEST:
    EPOCHS, MC_PASSES, XAI_SAMPLES = 1, 3, 4
FT_EPOCHS = (1 if QUICK_TEST else (EPOCHS if FT_EPOCHS is None else FT_EPOCHS))
FT_LR     = LR if FT_LR is None else FT_LR
ALL_MODELS = list(BATCH_SIZES)
MODELS = ALL_MODELS if MODELS == "all" else list(MODELS)
if QUICK_TEST:
    MODELS = MODELS[:1]
bad = [m for m in MODELS if m not in BATCH_SIZES]
assert not bad, f"Unknown model {bad}; valid: {ALL_MODELS}"
assert TRAIN_DATASETS == ["figshare"], "TRAIN_DATASETS must be [\"figshare\"]: pcosgen is stage 2 (FINETUNE_FIGSHARE_TO_PCOSGEN), not a separate training."
assert FINETUNE_FIGSHARE_TO_PCOSGEN, "Set FINETUNE_FIGSHARE_TO_PCOSGEN = True to run stage 2 (pcosgen)."

### A2. Mount Drive, GPU, unpack code

In [39]:
import os, sys, re, shutil, subprocess, json, time, zipfile, base64, io
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"          # Colab's inline backend breaks plain venvs
os.environ.pop("PYTHONPATH", None)

from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

DRIVE = Path(DRIVE_DIR)
WORK  = Path("/content/pearl")
assert DRIVE.exists(), f"Drive folder not found: {DRIVE}"
WORK.mkdir(parents=True, exist_ok=True)

# experiment name = which preprocessing variant (data_gauss.zip -> gauss, data_srad.zip -> srad)
EXP = EXP_NAME or re.sub(r"^data[_-]?", "", re.sub(r"\.zip$", "", DATA_ZIP))
if QUICK_TEST:
    EXP = "quicktest_" + EXP
print(f"Experiment: {EXP}   (data zip: {DATA_ZIP})")

g = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
                   capture_output=True, text=True)
if g.returncode != 0:
    raise RuntimeError("No GPU. Runtime > Change runtime type > pick a GPU")
GPU_NAME, GPU_MEM, GPU_CC = [x.strip() for x in g.stdout.strip().splitlines()[0].split(",")]
print(f"GPU: {GPU_NAME} | {GPU_MEM} | cc {GPU_CC}")
if PRECISION == "auto":
    PRECISION = "bf16" if float(GPU_CC) >= 8.0 else "fp16"
if PRECISION == "bf16" and float(GPU_CC) < 8.0:
    raise RuntimeError("bf16 needs Ampere+. Use PRECISION='fp16' on a T4.")
PREC_FLAGS = ["--bf16"] if PRECISION == "bf16" else ["--fp16"]
if NUM_WORKERS is None:
    NUM_WORKERS = min(8, os.cpu_count() or 2)
print(f"precision={PRECISION} workers={NUM_WORKERS}")
print("Drive:", sorted(p.name for p in DRIVE.iterdir()))

Experiment: srad   (data zip: data_srad.zip)
GPU: Tesla T4 | 15360 MiB | cc 7.5
precision=fp16 workers=2
Drive: ['data_gauss.zip', 'data_srad.zip', 'models', 'pearl_ensemble_standalone.ipynb', 'pearl_train_standalone.ipynb', 'pearl_venv_t4.tar.gz', 'results']


In [40]:
# Unpack the embedded code into /content/pearl (src/ and configs/). Leaves .venv and data/ alone.
for d in ("src", "configs"):
    if (WORK / d).exists():
        shutil.rmtree(WORK / d)
with zipfile.ZipFile(io.BytesIO(base64.b64decode("".join(BUNDLE_B64.split())))) as z:
    z.extractall(WORK)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("Code unpacked:", sum(1 for _ in (WORK / "src").rglob("*.py")), "modules,",
      sum(1 for _ in (WORK / "configs").rglob("*.yaml")), "configs")

Code unpacked: 49 modules, 9 configs


### A3. Restore venv from Drive

In [41]:
VENV_DIR = WORK / ".venv"
PY_DIR   = WORK / ".uv-python"
PY = str(VENV_DIR / "bin" / "python")

VERIFY = """
import torch, timm, torchcam, sklearn, cv2, yaml, optuna, PIL, rich
assert torch.cuda.is_available(), "CUDA not available in venv torch"
x = torch.randn(32, 32, device="cuda"); float((x @ x).sum())
print("venv OK | python", __import__("platform").python_version(), "| torch", torch.__version__, "| timm", timm.__version__, "|", torch.cuda.get_device_name(0))
"""

def venv_ok():
    if not Path(PY).exists():
        return False
    subprocess.run(f"chmod -R u+x {VENV_DIR}/bin", shell=True)
    r = subprocess.run([PY, "-c", VERIFY], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip()[-600:])
    return r.returncode == 0

t0 = time.time()
if not venv_ok():
    arc = DRIVE / VENV_ARCHIVE
    assert arc.exists(), f"{arc} not found on Drive."
    print(f"Unpacking {arc.name} ({arc.stat().st_size / 2**30:.2f} GB) ...")
    for d in (VENV_DIR, PY_DIR):
        if d.exists():
            shutil.rmtree(d)
    subprocess.run(["tar", "-C", str(WORK), "-xzf", str(arc)], check=True)
    assert venv_ok(), "venv from archive is not usable"
print(f"venv ready in {time.time() - t0:.0f}s -> {PY}")

venv OK | python 3.12.15 | torch 2.12.0+cu130 | timm 1.0.27 | Tesla T4
venv ready in 6s -> /content/pearl/.venv/bin/python


### A4. Unpack data

In [42]:
# Unpack the data zip (expects data/preprocessed/<figshare|pcosgen>/{train,val,test}). Re-extracts if DATA_ZIP changed.
DATA_MARK = WORK / "data" / ".data_zip_name"
if DATA_MARK.exists() and DATA_MARK.read_text() == DATA_ZIP:
    print(f"[SKIP] {DATA_ZIP} already unpacked.")
else:
    zp = DRIVE / DATA_ZIP
    assert zp.exists(), f"{zp} not found on Drive."
    if (WORK / "data").exists():
        shutil.rmtree(WORK / "data")
    print(f"Unzipping {zp.name} ({zp.stat().st_size / 2**20:.0f} MB) ...")
    r = subprocess.run(["unzip", "-q", "-o", str(zp), "-d", str(WORK)])
    assert r.returncode in (0, 1), "unzip failed"
    if not (WORK / "data" / "preprocessed").is_dir():          # zip had a wrapper folder
        for c in WORK.iterdir():
            if (c / "data" / "preprocessed").is_dir():
                shutil.move(str(c / "data"), str(WORK / "data")); break
    assert (WORK / "data" / "preprocessed").is_dir(), "data/preprocessed not found in zip"
    DATA_MARK.write_text(DATA_ZIP)

AVAILABLE_DS = {}
for ds in ("figshare", "pcosgen"):
    root = WORK / "data" / "preprocessed" / ds
    if [s for s in ("train", "val", "test") if not (root / s).is_dir()] == []:
        AVAILABLE_DS[ds] = root
        print(f"{ds}: " + ", ".join(f"{s}={sum(1 for _ in (root / s).rglob('*.png'))}" for s in ("train", "val", "test")))
    else:
        print(f"{ds}: MISSING")

[SKIP] data_srad.zip already unpacked.
figshare: train=2823, val=313, test=554
pcosgen: train=2560, val=640, test=1468


### A5. Helpers

In [43]:
def stream(cmd, cwd=WORK):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), "\n")
    env = dict(os.environ, PYTHONUNBUFFERED="1", MPLBACKEND="Agg")
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=str(cwd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print(f"\n[{'OK' if p.returncode == 0 else 'FAILED code ' + str(p.returncode)}] {(time.time() - t0) / 60:.1f} min")
    return p.returncode

def finished(d):
    d = Path(d)
    return (d / "best.pt").exists() and (d / "final_metrics.json").exists()

In [44]:
RES        = WORK / "results" / EXP
DRV_MODELS = DRIVE / "models" / EXP        # weights:  models/<exp>/<run>/<model>/best.pt ...
DRV_RESULTS= DRIVE / "results" / EXP       # metrics:  results/<exp>/<run>/<model>/...

# run = (tag, dataset trained on, init-from tag or None)
RUNS = [(ds, ds, None) for ds in TRAIN_DATASETS]
if FINETUNE_FIGSHARE_TO_PCOSGEN:
    assert "figshare" in TRAIN_DATASETS, "fine-tune needs a figshare run"
    RUNS.append(("pcosgen_ft", "pcosgen", "figshare"))
for _, ds, _ in RUNS:
    assert ds in AVAILABLE_DS, f"dataset {ds} is not in the data zip"
EVAL_SETS = list(AVAILABLE_DS)

def mdir(tag, m):
    return RES / tag / m

def save_to_drive(tag, m):
    # weights + run records -> models/ ; everything else (metrics, calibration, uncertainty, xai) -> results/
    if QUICK_TEST or not mdir(tag, m).exists():
        return
    src, dm, dr = mdir(tag, m), DRV_MODELS / tag / m, DRV_RESULTS / tag / m
    both = ("config.yaml", "final_metrics.json")
    for p in src.rglob("*"):
        if p.is_dir():
            continue
        rel = p.relative_to(src)
        top = len(rel.parts) == 1
        targets = []
        if top and (p.suffix in (".pt", ".pth") or p.name in both):
            targets.append(dm / rel)
        if not (top and p.suffix in (".pt", ".pth")):
            targets.append(dr / rel)
        for dst in targets:
            if dst.exists() and dst.stat().st_size == p.stat().st_size and dst.stat().st_mtime >= p.stat().st_mtime:
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dst)
    print(f"[SAVED] {tag}/{m} -> {dm}  and  {dr}")

def restore_from_drive():
    if QUICK_TEST:
        return
    for tag, _, _ in RUNS:
        for m in MODELS:
            d = DRV_MODELS / tag / m
            if finished(d) and not finished(mdir(tag, m)):
                shutil.copytree(d, mdir(tag, m), dirs_exist_ok=True)
                print(f"[RESTORED] {tag}/{m} from Drive")

restore_from_drive()
print("Runs:", [(t, f"train={d}", f"init={i}") for t, d, i in RUNS])
print("Models:", MODELS)
print("Drive output:", DRV_MODELS, "|", DRV_RESULTS)

Runs: [('figshare', 'train=figshare', 'init=None'), ('pcosgen_ft', 'train=pcosgen', 'init=figshare')]
Models: ['vit_base']
Drive output: /content/drive/MyDrive/thesis/models/srad | /content/drive/MyDrive/thesis/results/srad


## B. Train + evaluate + calibrate + save to Drive
Re-run-safe. A failing stage is reported and the rest continue. Everything is copied to Drive after each stage.

In [45]:
def train_run(tag, ds, init, m):
    if SKIP_TRAINED and finished(mdir(tag, m)):
        return "skipped (already trained)"
    cmd = [PY, "-m", "src.train", "--dataset-dir", f"data/preprocessed/{ds}", "--model", m,
           "--output-dir", f"results/{EXP}/{tag}", "--batch-size", BATCH_SIZES[m], "--seed", SEED,
           "--num-workers", NUM_WORKERS, *PREC_FLAGS, *EXTRA_TRAIN_ARGS]
    if init:
        if not finished(mdir(init, m)):
            return f"skipped (init model {init}/{m} not trained)"
        cmd += ["--checkpoint", f"results/{EXP}/{init}/{m}/best.pt", "--epochs", FT_EPOCHS, "--lr", FT_LR]
    else:
        cmd += ["--epochs", EPOCHS, "--lr", LR]
    rc = stream(cmd)
    if rc != 0:
        return f"FAILED (exit {rc}; if CUDA OOM lower BATCH_SIZES['{m}'])"
    save_to_drive(tag, m)
    return "ok"

def analyse_run(tag, ds, m):
    if not finished(mdir(tag, m)):
        print(f"{tag}/{m}: not trained, skipping analysis.")
        return {}
    d, ck, cfg, dsdir = f"results/{EXP}/{tag}/{m}", f"results/{EXP}/{tag}/{m}/best.pt", f"configs/model/{m}.yaml", f"data/preprocessed/{ds}"
    stages = []
    if RUN_EVALUATE:
        for t in EVAL_SETS:
            out = "metrics.json" if t == ds else f"cross_{t}.json"
            stages.append((f"evaluate on {t} ({'in-domain' if t == ds else 'cross-domain'})",
                           [PY, "-m", "src.evaluation.evaluate", "--test-dataset-dir", f"data/preprocessed/{t}/test",
                            "--checkpoint-dir", d, "--model-config", cfg, "--output", f"{d}/{out}"]))
    if RUN_CALIBRATION:
        stages.append(("calibration", [PY, "-m", "src.calibration.run_calibration", "--dataset-dir", dsdir,
                       "--model", m, "--checkpoint", ck, "--out-dir", f"{d}/calibration"]))
    if RUN_UNCERTAINTY:
        stages.append(("uncertainty", [PY, "-m", "src.uncertainty.run_uncertainty", "--dataset-dir", dsdir,
                       "--model", m, "--checkpoint", ck, "--out-dir", f"{d}/uncertainty", "--mc-passes", MC_PASSES]))
    if RUN_XAI:
        stages.append(("xai", [PY, "-m", "src.xai.run_xai", "--dataset-dir", dsdir, "--model", m, "--checkpoint", ck,
                       "--out-dir", f"{d}/xai", "--methods", *XAI_METHODS, "--n-samples", XAI_SAMPLES, "--mc-passes", MC_PASSES]))
    status = {}
    for name, cmd in stages:
        print("\n" + "-" * 70 + f"\n{m} [{tag}]: {name}\n" + "-" * 70)
        t0 = time.time()
        status[name] = (stream(cmd), (time.time() - t0) / 60)
        save_to_drive(tag, m)            # save after every stage so a disconnect loses nothing
    return status

def summarize():
    rows = {}
    for tag, ds, _ in RUNS:
        for m in MODELS:
            for f in sorted(mdir(tag, m).glob("*.json")) if mdir(tag, m).exists() else []:
                if f.name in ("final_metrics.json",):
                    continue
                try:
                    j = json.loads(f.read_text())
                except Exception:
                    continue
                keep = {k: round(v, 4) for k, v in j.items() if isinstance(v, (int, float)) and not isinstance(v, bool)
                        and any(s in k.lower() for s in ("auc", "accuracy", "f1", "recall", "specificity", "ece"))}
                rows[f"{tag}/{m}/{f.stem}"] = keep
    for k, v in rows.items():
        print(f"{k}\n   {v}")
    if not QUICK_TEST and rows:
        DRV_RESULTS.mkdir(parents=True, exist_ok=True)
        (DRV_RESULTS / "summary_train.json").write_text(json.dumps(rows, indent=2))
        print(f"\n[SAVED] summary -> {DRV_RESULTS / 'summary_train.json'}")
    return rows

def run_all():
    status = {}
    for m in MODELS:
        for tag, ds, init in RUNS:
            print("\n" + "=" * 70 + f"\nTRAIN  model={m}  run={tag}  (data={ds}, init={init or 'ImageNet'})\n" + "=" * 70)
            status[f"{m} [{tag}] train"] = train_run(tag, ds, init, m)
            for name, (rc, mins) in analyse_run(tag, ds, m).items():
                status[f"{m} [{tag}] {name}"] = f"{'OK' if rc == 0 else 'FAILED'} ({mins:.1f} min)"
            save_to_drive(tag, m)
    print("\n" + "=" * 70 + "\nSTATUS\n" + "=" * 70)
    for k, v in status.items():
        print(f"  {k:60s} {v}")
    print("\nRESULTS\n")
    summarize()
    if QUICK_TEST:
        print("\nQUICK_TEST done (nothing was saved to Drive). Set QUICK_TEST=False, re-run the config + setup cells, then run this cell.")
    return status

run_all()


TRAIN  model=vit_base  run=figshare  (data=figshare, init=ImageNet)
$ /content/pearl/.venv/bin/python -m src.train --dataset-dir data/preprocessed/figshare --model vit_base --output-dir results/srad/figshare --batch-size 16 --seed 42 --num-workers 2 --fp16 --epochs 100 --lr 0.0001 

[Data] dataset=data/preprocessed/figshare train=2823 val=313 test=554
[Builder] Frozen 90/150 parameters (60.0%)
[Builder] Total params: 86,029,442, Trainable: 35,670,146
[Loss] Class weights: [2.3292078971862793, 0.6366711854934692]  label_smoothing=0.05
[Logger] Run directory: results/srad/figshare/vit_base (resume=False)

Training for up to 100 epochs (patience=20, warmup=2)

  Epoch   1 | train_loss=0.2436  val_loss=0.3648 | val_acc=0.9681  val_auc=0.9893  val_f1=0.9798  val_mcc=0.9036 | lr=5.00e-05  time=18.4s
[Checkpoint] Saved (epoch=1, val_auc=0.9893) → results/srad/figshare/vit_base/best.pt
  Epoch   2 | train_loss=0.2067  val_loss=0.3266 | val_acc=0.9553  val_auc=0.9826  val_f1=0.9712  val_mcc=0.

{'vit_base [figshare] train': 'ok',
 'vit_base [figshare] evaluate on figshare (in-domain)': 'OK (0.4 min)',
 'vit_base [figshare] evaluate on pcosgen (cross-domain)': 'OK (0.6 min)',
 'vit_base [figshare] calibration': 'OK (0.5 min)',
 'vit_base [figshare] uncertainty': 'OK (6.0 min)',
 'vit_base [pcosgen_ft] train': 'ok',
 'vit_base [pcosgen_ft] evaluate on figshare (cross-domain)': 'OK (0.4 min)',
 'vit_base [pcosgen_ft] evaluate on pcosgen (in-domain)': 'OK (0.5 min)',
 'vit_base [pcosgen_ft] calibration': 'OK (0.7 min)',
 'vit_base [pcosgen_ft] uncertainty': 'OK (15.2 min)'}